In [1]:
import pandas as pd
import numpy as np
#import skfuzzy as fuzz  
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
df= pd.read_csv("C:\\Users\\MyMachine\\Desktop\\Fuzzy-Logic-Based-Maternal-Mental-Health-Risk-Assessment\\Data_Synthesis\\sampled_maternal_data_20k.csv")

In [3]:
df.columns

Index(['age', 'residence', 'province', 'ethnicity',
       'caste_discrimination_exposure', 'education_level', 'occupation',
       'family_type', 'wealth_index', 'monthly_household_income',
       'food_security', 'housing_quality', 'financial_stress_level', 'parity',
       'anc_visits', 'delivery_type', 'birth_complications',
       'baby_health_status', 'gestational_age', 'maternal_age_at_first_birth',
       'previous_pregnancy_loss', 'partner_support_score',
       'family_support_score', 'domestic_violence_exposure',
       'social_isolation_score', 'sleep_quality', 'stressful_life_events',
       'mental_health_awareness', 'distance_to_health_facility',
       'health_insurance', 'previous_mental_health_consultation',
       'ppd_risk_level'],
      dtype='object')

In [4]:
df.shape

(20000, 32)

In [5]:
import pandas as pd
from sklearn.preprocessing import OneHotEncoder, LabelEncoder, OrdinalEncoder

In [6]:
onehot_features = [
    'province',
    'ethnicity',
    'occupation',
    'delivery_type',
    'baby_health_status',
    'mental_health_awareness'
]

In [7]:
binary_features = [
    'residence',
    'caste_discrimination_exposure',
    'family_type',
    'birth_complications',
    'previous_pregnancy_loss',
    'health_insurance',
    'previous_mental_health_consultation'
]

In [8]:
target_feature = 'ppd_risk_level'

ppd_order = [['No/Minimal Risk', 'Possible Depression', 'Probable Depression']]

In [9]:
onehot_features

['province',
 'ethnicity',
 'occupation',
 'delivery_type',
 'baby_health_status',
 'mental_health_awareness']

In [10]:
df["province"].unique()

array(['Koshi', 'Madhesh', 'Sudurpashchim', 'Gandaki', 'Lumbini',
       'Bagmati', 'Karnali'], dtype=object)

In [11]:
onehot_encoder = OneHotEncoder(sparse_output=False, drop='first')

onehot_encoded_array = onehot_encoder.fit_transform(df[onehot_features])

onehot_encoded_df = pd.DataFrame(
    onehot_encoded_array,
    columns=onehot_encoder.get_feature_names_out(onehot_features),
    index=df.index
)

In [12]:
df_binary_encoded = df[binary_features].copy()
df_binary_encoded

,residence,caste_discrimination_exposure,family_type,birth_complications,previous_pregnancy_loss,health_insurance,previous_mental_health_consultation
0,Urban,No,Nuclear,No,No,Yes,Yes
1,Rural,Yes,Nuclear,Yes,No,No,No
2,Rural,No,Nuclear,No,No,No,No
3,Rural,No,Joint/Extended,No,No,No,No
4,Rural,No,Nuclear,No,No,No,No
...,...,...,...,...,...,...,...
19995,Rural,No,Joint/Extended,No,Yes,No,No
19996,Urban,No,Nuclear,No,No,Yes,No
19997,Urban,No,Nuclear,No,Yes,No,No
19998,Rural,No,Joint/Extended,No,No,No,No


In [13]:
label_encoders = {}

for col in binary_features:
    le = LabelEncoder()
    df_binary_encoded[col] = le.fit_transform(df_binary_encoded[col])
    label_encoders[col] = le

In [14]:
ordinal_encoder = OrdinalEncoder(categories=ppd_order)

df_target_encoded = pd.DataFrame(
    ordinal_encoder.fit_transform(df[[target_feature]]).astype(int),
    columns=[target_feature],
    index=df.index
)

In [15]:
# Get the mapping of the target feature ppd_risk_level 

ppd_risk_mapping = {level: code for code, level in enumerate(df['ppd_risk_level'].unique())}
print(ppd_risk_mapping)

# Apply the mapping to the target feature
df['ppd_risk_level'] = df['ppd_risk_level'].map(ppd_risk_mapping)

{'No/Minimal Risk': 0, 'Probable Depression': 1, 'Possible Depression': 2}


In [16]:
df_crisp_encoded = pd.concat(
    [onehot_encoded_df, df_binary_encoded, df_target_encoded],
    axis=1
)

In [17]:
df_crisp_encoded.columns

Index(['province_Gandaki', 'province_Karnali', 'province_Koshi',
       'province_Lumbini', 'province_Madhesh', 'province_Sudurpashchim',
       'ethnicity_Hill High Caste', 'ethnicity_Janajati', 'ethnicity_Muslim',
       'ethnicity_Terai Caste', 'occupation_Clerical', 'occupation_Manual',
       'occupation_Professional/Technical/Managerial',
       'occupation_Sales/Service', 'delivery_type_Cesarean',
       'delivery_type_Normal', 'baby_health_status_Major issues',
       'baby_health_status_Minor issues', 'mental_health_awareness_Low',
       'mental_health_awareness_Medium', 'residence',
       'caste_discrimination_exposure', 'family_type', 'birth_complications',
       'previous_pregnancy_loss', 'health_insurance',
       'previous_mental_health_consultation', 'ppd_risk_level'],
      dtype='object')

> Nominal categorical variables (province, ethnicity, occupation, delivery type, baby health status, and mental health awareness) were transformed using **one-hot encoding**. Dichotomous predictors (residence, caste discrimination exposure, family type, birth complications, previous pregnancy loss, health insurance, and previous mental health consultation) were encoded using **binary label encoding (0–1)**. The target variable *ppd_risk_level*, representing ordered clinical severity, was transformed using **ordinal encoding (0–2)**. These crisp-transformed features were later concatenated with fuzzified continuous and ordinal predictors to construct the final hybrid fuzzy–crisp feature matrix.

In [18]:
df_crisp_encoded

,province_Gandaki,province_Karnali,province_Koshi,province_Lumbini,province_Madhesh,province_Sudurpashchim,ethnicity_Hill High Caste,ethnicity_Janajati,ethnicity_Muslim,ethnicity_Terai Caste,...,mental_health_awareness_Low,mental_health_awareness_Medium,residence,caste_discrimination_exposure,family_type,birth_complications,previous_pregnancy_loss,health_insurance,previous_mental_health_consultation,ppd_risk_level
0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,...,0.0,1.0,1,0,1,0,0,1,1,0
1,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,1.0,0.0,0,1,1,1,0,0,0,2
2,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,...,0.0,0.0,0,0,1,0,0,0,0,0
3,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,...,0.0,1.0,0,0,0,0,0,0,0,0
4,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,...,0.0,1.0,0,0,1,0,0,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19995,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,...,0.0,1.0,0,0,0,0,1,0,0,0
19996,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,...,0.0,1.0,1,0,1,0,0,1,0,0
19997,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,...,0.0,1.0,1,0,1,0,1,0,0,0
19998,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,...,0.0,1.0,0,0,0,0,0,0,0,1


In [19]:
df_crisp_encoded.columns

Index(['province_Gandaki', 'province_Karnali', 'province_Koshi',
       'province_Lumbini', 'province_Madhesh', 'province_Sudurpashchim',
       'ethnicity_Hill High Caste', 'ethnicity_Janajati', 'ethnicity_Muslim',
       'ethnicity_Terai Caste', 'occupation_Clerical', 'occupation_Manual',
       'occupation_Professional/Technical/Managerial',
       'occupation_Sales/Service', 'delivery_type_Cesarean',
       'delivery_type_Normal', 'baby_health_status_Major issues',
       'baby_health_status_Minor issues', 'mental_health_awareness_Low',
       'mental_health_awareness_Medium', 'residence',
       'caste_discrimination_exposure', 'family_type', 'birth_complications',
       'previous_pregnancy_loss', 'health_insurance',
       'previous_mental_health_consultation', 'ppd_risk_level'],
      dtype='object')

In [20]:
df_crisp_encoded.shape

(20000, 28)

In [21]:
df.columns

Index(['age', 'residence', 'province', 'ethnicity',
       'caste_discrimination_exposure', 'education_level', 'occupation',
       'family_type', 'wealth_index', 'monthly_household_income',
       'food_security', 'housing_quality', 'financial_stress_level', 'parity',
       'anc_visits', 'delivery_type', 'birth_complications',
       'baby_health_status', 'gestational_age', 'maternal_age_at_first_birth',
       'previous_pregnancy_loss', 'partner_support_score',
       'family_support_score', 'domestic_violence_exposure',
       'social_isolation_score', 'sleep_quality', 'stressful_life_events',
       'mental_health_awareness', 'distance_to_health_facility',
       'health_insurance', 'previous_mental_health_consultation',
       'ppd_risk_level'],
      dtype='object')

In [22]:
df.shape

(20000, 32)

In [23]:
# Delete one-hot-encoding original features: -6
df.drop(columns= onehot_features, inplace=True)

In [24]:
df.shape

(20000, 26)

In [25]:
# delete binary encoded original features: -7 
df.drop(columns= binary_features, inplace=True)

In [26]:
df.shape

(20000, 19)

In [27]:
# delete target column:: -1
df.drop(columns= target_feature, inplace=True)

In [28]:
df.shape

(20000, 18)

In [29]:
df.columns

Index(['age', 'education_level', 'wealth_index', 'monthly_household_income',
       'food_security', 'housing_quality', 'financial_stress_level', 'parity',
       'anc_visits', 'gestational_age', 'maternal_age_at_first_birth',
       'partner_support_score', 'family_support_score',
       'domestic_violence_exposure', 'social_isolation_score', 'sleep_quality',
       'stressful_life_events', 'distance_to_health_facility'],
      dtype='object')

In [30]:
# concat df and df_crisp_encoded columns to create a complete featurs matrix: 
df_mat= pd.concat([df, df_crisp_encoded], axis=1)

In [31]:
df_mat.shape

(20000, 46)

In [32]:
df_mat.columns

Index(['age', 'education_level', 'wealth_index', 'monthly_household_income',
       'food_security', 'housing_quality', 'financial_stress_level', 'parity',
       'anc_visits', 'gestational_age', 'maternal_age_at_first_birth',
       'partner_support_score', 'family_support_score',
       'domestic_violence_exposure', 'social_isolation_score', 'sleep_quality',
       'stressful_life_events', 'distance_to_health_facility',
       'province_Gandaki', 'province_Karnali', 'province_Koshi',
       'province_Lumbini', 'province_Madhesh', 'province_Sudurpashchim',
       'ethnicity_Hill High Caste', 'ethnicity_Janajati', 'ethnicity_Muslim',
       'ethnicity_Terai Caste', 'occupation_Clerical', 'occupation_Manual',
       'occupation_Professional/Technical/Managerial',
       'occupation_Sales/Service', 'delivery_type_Cesarean',
       'delivery_type_Normal', 'baby_health_status_Major issues',
       'baby_health_status_Minor issues', 'mental_health_awareness_Low',
       'mental_health_aware

In [33]:
df_mat["ppd_risk_level"].value_counts()

ppd_risk_level
0    8149
1    8092
2    3759
Name: count, dtype: int64

In [34]:
df_mat.columns

Index(['age', 'education_level', 'wealth_index', 'monthly_household_income',
       'food_security', 'housing_quality', 'financial_stress_level', 'parity',
       'anc_visits', 'gestational_age', 'maternal_age_at_first_birth',
       'partner_support_score', 'family_support_score',
       'domestic_violence_exposure', 'social_isolation_score', 'sleep_quality',
       'stressful_life_events', 'distance_to_health_facility',
       'province_Gandaki', 'province_Karnali', 'province_Koshi',
       'province_Lumbini', 'province_Madhesh', 'province_Sudurpashchim',
       'ethnicity_Hill High Caste', 'ethnicity_Janajati', 'ethnicity_Muslim',
       'ethnicity_Terai Caste', 'occupation_Clerical', 'occupation_Manual',
       'occupation_Professional/Technical/Managerial',
       'occupation_Sales/Service', 'delivery_type_Cesarean',
       'delivery_type_Normal', 'baby_health_status_Major issues',
       'baby_health_status_Minor issues', 'mental_health_awareness_Low',
       'mental_health_aware

In [35]:
df_mat

,age,education_level,wealth_index,monthly_household_income,food_security,housing_quality,financial_stress_level,parity,anc_visits,gestational_age,...,mental_health_awareness_Low,mental_health_awareness_Medium,residence,caste_discrimination_exposure,family_type,birth_complications,previous_pregnancy_loss,health_insurance,previous_mental_health_consultation,ppd_risk_level
0,21,Secondary,Richer,51044,Secure,9,0,0,8,40,...,0.0,1.0,1,0,1,0,0,1,1,0
1,38,Primary,Richer,39767,Moderately insecure,8,1,2,7,35,...,1.0,0.0,0,1,1,1,0,0,0,2
2,38,Secondary,Middle,45000,Mildly insecure,5,4,1,2,39,...,0.0,0.0,0,0,1,0,0,0,0,0
3,41,Secondary,Poorer,17768,Mildly insecure,4,6,0,6,38,...,0.0,1.0,0,0,0,0,0,0,0,0
4,43,Primary,Richer,61897,Secure,8,4,3,10,40,...,0.0,1.0,0,0,1,0,0,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19995,27,Secondary,Middle,23618,Mildly insecure,6,7,1,8,40,...,0.0,1.0,0,0,0,0,1,0,0,0
19996,26,Primary,Poorer,22317,Moderately insecure,5,7,3,6,40,...,0.0,1.0,1,0,1,0,0,1,0,0
19997,41,Primary,Poorest,17945,Mildly insecure,4,6,3,6,38,...,0.0,1.0,1,0,1,0,1,0,0,0
19998,17,Secondary,Middle,32066,Mildly insecure,5,6,0,11,37,...,0.0,1.0,0,0,0,0,0,0,0,1


In [34]:
# save df_mat as a csv file: 
df_mat.to_csv('df_mat.csv', index=False)

In [37]:
df_mat.shape

(20000, 46)

In [35]:
df_mat

,age,education_level,wealth_index,monthly_household_income,food_security,housing_quality,financial_stress_level,parity,anc_visits,gestational_age,...,mental_health_awareness_Low,mental_health_awareness_Medium,residence,caste_discrimination_exposure,family_type,birth_complications,previous_pregnancy_loss,health_insurance,previous_mental_health_consultation,ppd_risk_level
0,21,Secondary,Richer,51044,Secure,9,0,0,8,40,...,0.0,1.0,1,0,1,0,0,1,1,0
1,38,Primary,Richer,39767,Moderately insecure,8,1,2,7,35,...,1.0,0.0,0,1,1,1,0,0,0,2
2,38,Secondary,Middle,45000,Mildly insecure,5,4,1,2,39,...,0.0,0.0,0,0,1,0,0,0,0,0
3,41,Secondary,Poorer,17768,Mildly insecure,4,6,0,6,38,...,0.0,1.0,0,0,0,0,0,0,0,0
4,43,Primary,Richer,61897,Secure,8,4,3,10,40,...,0.0,1.0,0,0,1,0,0,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19995,27,Secondary,Middle,23618,Mildly insecure,6,7,1,8,40,...,0.0,1.0,0,0,0,0,1,0,0,0
19996,26,Primary,Poorer,22317,Moderately insecure,5,7,3,6,40,...,0.0,1.0,1,0,1,0,0,1,0,0
19997,41,Primary,Poorest,17945,Mildly insecure,4,6,3,6,38,...,0.0,1.0,1,0,1,0,1,0,0,0
19998,17,Secondary,Middle,32066,Mildly insecure,5,6,0,11,37,...,0.0,1.0,0,0,0,0,0,0,0,1


In [36]:
df_mat["mental_health_awareness_Low"]

0        0.0
1        1.0
2        0.0
3        0.0
4        0.0
        ... 
19995    0.0
19996    0.0
19997    0.0
19998    0.0
19999    0.0
Name: mental_health_awareness_Low, Length: 20000, dtype: float64